[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_fuga_escuela.ipynb)

# Fuga de información: el examen final

**La situación.** Una escuela quiere saber, **un mes antes del examen final**, qué alumnos van a salir mal,
para darles asesoría a tiempo. Tiene datos de 300 alumnos del año pasado.

**La pregunta de toda la hoja:** un mes antes del examen, **¿qué datos ya tengo?**
Si el modelo usa algo que todavía no existe en ese momento, está haciendo trampa: eso es **fuga de información**.

Corran las celdas en orden. No hace falta Colab especial: solo pandas y scikit-learn.

### Celda 1. Los datos (inventados, pero realistas)

In [ ]:
import random
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

random.seed(0)
filas = []
for i in range(300):
    horas = random.randint(0, 10)                      # horas de estudio por semana
    asistencia = random.randint(50, 100)               # % de clases a las que fue
    tareas = random.randint(4, 10)                     # promedio de tareas
    examen = 2 + 0.4 * horas + 0.03 * asistencia + 0.2 * tareas + random.gauss(0, 0.8)
    examen = max(0, min(10, round(examen, 1)))         # calificación del examen final, de 0 a 10
    final = round(0.5 * tareas + 0.5 * examen, 1)      # calificación final del curso: mitad tareas, mitad examen
    filas.append({"horas": horas, "asistencia": asistencia, "tareas": tareas,
                  "examen": examen, "final": final})
alumnos = pd.DataFrame(filas)
alumnos.head()

**Qué es cada columna y cuándo se conoce:**

| Columna | Qué es | ¿La tengo un mes antes del examen? |
|---|---|---|
| `horas` | horas de estudio por semana | **Sí** |
| `asistencia` | % de clases a las que fue | **Sí** |
| `tareas` | promedio de tareas | **Sí** |
| `examen` | calificación del examen final: **lo que queremos predecir** | No: es la respuesta |
| `final` | calificación final del curso: mitad tareas, **mitad examen** | **No:** se calcula después del examen |

### Celda 2. Partir: 200 alumnos para aprender, 100 para medir

In [ ]:
train = alumnos.iloc[:200]    # con estos aprende el modelo
prueba = alumnos.iloc[200:]   # con estos medimos: el modelo no los vio

def mide(columnas):
    modelo = LinearRegression().fit(train[columnas], train.examen)
    error = mean_absolute_error(prueba.examen, modelo.predict(prueba[columnas]))
    print(f"se equivoca por {error:.2f} puntos   columnas: {columnas}")

### Celda 3. El modelo honesto
Solo con lo que la escuela sabe un mes antes: horas de estudio, asistencia y tareas.

In [ ]:
honesto = ["horas", "asistencia", "tareas"]
mide(honesto)

Se equivoca por unos **0.6 puntos** (de 10). Es un modelo útil: alcanza para saber quién va en riesgo.

### Celda 4. Le agrego la calificación final del curso
Alguien dice: "la calificación final tiene mucho que ver con el examen, agréguenla".

In [ ]:
mide(honesto + ["final"])

Se equivoca por **casi cero**. ¿Ya resolvimos el problema? **No.** La calificación final **se calcula con el examen**:
es mitad tareas y mitad examen. Si tengo la final y las tareas, el examen sale solo con una cuenta.
Y lo más importante: **un mes antes del examen, la calificación final no existe.** El modelo es perfecto en la computadora
e inútil en la vida real: no se puede usar para lo que queríamos, avisar a tiempo.

### Celda 5. La solución: preguntar, columna por columna, cuándo existe

In [ ]:
cuando_existe = {"horas": "antes", "asistencia": "antes", "tareas": "antes", "final": "después"}

def mide_seguro(columnas):
    tramposas = [c for c in columnas if cuando_existe[c] == "después"]
    if tramposas:
        print("ALTO: estas columnas no existen un mes antes del examen:", tramposas)
        return
    mide(columnas)

mide_seguro(honesto + ["final"])   # se detiene
mide_seguro(honesto)               # corre: es el modelo honesto

## Para llevar
1. **¿Cuándo existe cada columna?** Si existe hasta después del momento en que hay que decidir, fuera.
2. **Si el error es casi cero, sospechen.** En la vida real casi nada se predice perfecto.
3. **Medir en datos nuevos no descubre la fuga:** la columna tramposa también está en esos datos. La descubren ustedes, preguntando cuándo existe cada columna.

**Lo mismo en MiBici:** las llegadas de 8 a 9 son como la calificación final: pasan al mismo tiempo que lo que queremos predecir.